# 02 - Treniranje TF-IDF + Logistic Regression modela

Ova sveska prati skriptu `scripts/03_train_tfidf_logreg.py`. Tekst mejla se prvo pretvara u numericke atribute pomocu TF-IDF vektorizacije, a zatim se trenira logisticka regresija.

Model se bira na validacionom skupu. Test skup se ovde ne koristi, jer se cuva za finalnu procenu.

In [ ]:
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

ROOT

## Ucitavanje trening i validacionog skupa

In [ ]:
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline

TRAIN_PATH = ROOT / "data" / "processed" / "splits" / "train.csv"
VALIDATION_PATH = ROOT / "data" / "processed" / "splits" / "validation.csv"
MODEL_PATH = ROOT / "models" / "tfidf_logreg_pipeline.joblib"
RESULTS_PATH = ROOT / "reports" / "03_tfidf_logreg_results.txt"

train_df = pd.read_csv(TRAIN_PATH)
validation_df = pd.read_csv(VALIDATION_PATH)

x_train = train_df["message"]
y_train = train_df["label"]
x_validation = validation_df["message"]
y_validation = validation_df["label"]

train_df.head()

## Generisanje kandidata hiperparametara

Menjamo `min_df` i regularizacioni parametar `C`. Ostale vrednosti drzimo fiksnim da pretraga ostane jednostavna.

In [ ]:
candidate_parameters = [
    (min_df_param, c_param)
    for min_df_param in [2, 5, 10]
    for c_param in [0.5, 1.0, 2.0, 5.0]
]

print(f"Broj kandidata: {len(candidate_parameters)}")
print(candidate_parameters)

## Funkcija koja pravi model na osnovu parametara

Koristimo `Pipeline`, jer on vezuje TF-IDF transformaciju i logisticku regresiju u jedan model.

In [ ]:
def napravi_model(min_df_param, c_param):
    return Pipeline(
        [
            (
                "tfidf",
                TfidfVectorizer(
                    lowercase=True,
                    strip_accents="unicode",
                    ngram_range=(1, 2),
                    min_df=min_df_param,
                    max_df=0.90,
                    max_features=10000,
                    sublinear_tf=True,
                ),
            ),
            (
                "logistic_regression",
                LogisticRegression(
                    C=c_param,
                    max_iter=1000,
                    class_weight="balanced",
                    random_state=42,
                    solver="liblinear",
                ),
            ),
        ]
    )

## Treniranje kandidata i izbor najboljeg modela

Za svaki par hiperparametara treniramo model na trening skupu i merimo F1 za `spam` i `ham` na validacionom skupu. Primarni fokus je spam klasa.

In [ ]:
all_results = []
best_model = None
best_params = None
best_spam_f1 = -1
best_ham_f1 = -1

for min_df_param, c_param in candidate_parameters:
    candidate_model = napravi_model(min_df_param, c_param)
    candidate_model.fit(x_train, y_train)
    predictions = candidate_model.predict(x_validation)

    spam_f1 = f1_score(y_validation, predictions, pos_label="spam", zero_division=0)
    ham_f1 = f1_score(y_validation, predictions, pos_label="ham", zero_division=0)

    all_results.append([min_df_param, c_param, spam_f1, ham_f1])

    if spam_f1+ham_f1> best_spam_f1+best_ham_f1:
        best_spam_f1 = spam_f1
        best_ham_f1 = ham_f1
        best_params = (min_df_param, c_param)
        best_model = candidate_model

results_table = pd.DataFrame(all_results, columns=["min_df", "C", "spam_f1", "ham_f1"]).sort_values("spam_f1", ascending=False)
results_table.head(10)

## Ispis rezultata najboljeg modela na validacionom skupu

In [ ]:
validation_predictions = best_model.predict(x_validation)
validation_report = classification_report(y_validation, validation_predictions)
validation_confusion_matrix = confusion_matrix(y_validation, validation_predictions, labels=["ham", "spam"])

print(f"Najbolji parametri: {best_params}")
print(validation_report)

pd.DataFrame(
    validation_confusion_matrix,
    index=["stvarno_ham", "stvarno_spam"],
    columns=["pred_ham", "pred_spam"],
)

## Cuvanje modela i rezultata

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, MODEL_PATH)

with open(RESULTS_PATH, "w", encoding="utf-8") as file:
    file.write("IZBOR HIPERPARAMETARA\n")
    file.write(str(results_table))
    file.write(f"\n\nNajbolji parametri: {best_params}\n\n")
    file.write("VALIDACIONI SKUP\n")
    file.write(validation_report)
    file.write("\n\nMATRICA KONFUZIJE NA VALIDACIONOM SKUPU\n")
    file.write(str(pd.DataFrame(validation_confusion_matrix, index=["stvarno_ham", "stvarno_spam"], columns=["pred_ham", "pred_spam"])))

print(f"Model je sacuvan u: {MODEL_PATH}")
print(f"Rezultati su sacuvani u: {RESULTS_PATH}")